# Gemini Live API + LangGraph Integration Pattern

This notebook demonstrates the recommended architecture: **Gemini Live API acts as the real-time voice interface**, and it triggers **LangGraph as a Tool/Function** when it needs to perform complex backend tasks (like saving data, running evaluations, or fetching from a database).

In [ ]:
!pip install google-genai python-dotenv

In [ ]:
import os
import asyncio
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load .env from backend directory to get GEMINI_API_KEY
load_dotenv("../.env")

client = genai.Client()

In [ ]:
# ---------------------------------------------------------
# 1. Define the LangGraph Entry Point (The "Tool")
# ---------------------------------------------------------
def trigger_langgraph_workflow(action: str, candidate_name: str) -> str:
    """
    Executes a complex background workflow via LangGraph.
    Use this tool when the user asks to save an evaluation, generate a report, or do deep analysis.
    
    Args:
        action: The action to perform (e.g. 'generate_report', 'save_evaluation')
        candidate_name: The name of the candidate.
    """
    print(f"\n\n[⚙️ SYSTEM] >>> LangGraph Triggered! Executing '{action}' for '{candidate_name}'...")
    
    # In a real app, you would invoke your graph here:
    # result = graph.invoke({"messages": [...], "intent": action})
    
    print("[⚙️ SYSTEM] >>> LangGraph workflow complete. Returning result to Gemini...")
    return f"Successfully completed {action} for {candidate_name} via LangGraph."



In [ ]:
# ---------------------------------------------------------
# 2. The Gemini Live Session
# ---------------------------------------------------------
async def run_live_with_tools():
    model_name = "gemini-2.0-flash-exp"
    
    # Configure the session to use our LangGraph tool
    config = types.LiveConnectConfig(
        response_modalities=[types.LiveModality.AUDIO], # Or TEXT if you just want to see text
        tools=[trigger_langgraph_workflow],
        system_instruction=types.Content(parts=[types.Part.from_text(
            "You are an HR Interviewer. You handle the conversational flow with the candidate. "
            "However, if the user asks you to generate a final report, save an evaluation, or do backend tasks, "
            "you MUST use the 'trigger_langgraph_workflow' tool. Once the tool returns a result, summarize it for the user verbally."
        )])
    )
    
    print("Connecting to Gemini Live...\n")
    async with client.aio.live.connect(model=model_name, config=config) as session:
        print("Connected!\n")
        
        # 1. We send a message that SHOULD trigger the tool
        user_message = "Hi, we just finished the interview for Amit. Can you generate the final report for him?"
        print(f"👤 User: {user_message}")
        
        await session.send(input=user_message, end_of_turn=True)
        
        # 2. Listen to Gemini's responses
        async for response in session.receive():
            if response.server_content is not None:
                model_turn = response.server_content.model_turn
                if model_turn is not None:
                    for part in model_turn.parts:
                        if part.text:
                            print(part.text, end="")
                        
                        elif part.inline_data:
                            # We are receiving AUDIO packets here
                            pass 
                        
                        elif part.function_call:
                            # GEMINI DECIDED TO USE OUR TOOL!
                            fc = part.function_call
                            print(f"\n\n🤖 Gemini requested tool call: {fc.name}() with arguments: {fc.args}")
                            
                            if fc.name == "trigger_langgraph_workflow":
                                # Extract the arguments Gemini generated
                                action = fc.args.get("action")
                                name = fc.args.get("candidate_name")
                                
                                # 3. Execute our local Python function (LangGraph)
                                tool_result = trigger_langgraph_workflow(action, name)
                                
                                # 4. Send the result back to the Live Session
                                # This tells Gemini what LangGraph did, so Gemini can reply to the user.
                                function_resp = types.FunctionResponse(
                                    name=fc.name,
                                    id=fc.id,
                                    response={"result": tool_result}
                                )
                                
                                await session.send(
                                    input=types.LiveClientContent(
                                        function_responses=[function_resp],
                                        turn_complete=True # Handing the turn back to Gemini
                                    )
                                )
                                
            # Break early after the model finishes its final turn for prototype purposes
            if response.server_content and response.server_content.turn_complete:
                print("\n\n[Turn Complete]")
                break


In [ ]:
# Run the async workflow
await run_live_with_tools()